# Marmousi2 Acoustic bv1.2 Inversion Validation

This notebook validates the inversion side of the Marmousi2 acoustic case. Run `01_forward_modeling.ipynb` first so the model, survey, and wavelet setup have been checked.

## 1. Paths And Imports

The inversion uses synthetic-true observations generated by the current code, then starts from `init_model.npz`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

CASE_DIR = REPO_ROOT / "examples" / "acoustic" / "01-model-test" / "01-Marmousi2"
VALIDATION_DIR = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12"
SCRIPT = VALIDATION_DIR / "scripts" / "run_validation.py"
OUTPUT_ROOT = VALIDATION_DIR / "outputs"

sys.path.insert(0, str(REPO_ROOT / "scripts" / "examples"))
from marmousi2_acoustic_backend_check import build_survey, load_npz

CASE_DIR

## 2. Parameter Definitions

These values mirror the current short Marmousi2 bv1.2 validation path. The 100-iteration stage should only be run after inspecting the 10-iteration output.

In [ ]:
INVERSION_CONFIG = {
    "device": "npu:0",
    "dtype": "float32",
    "true_model_file": "true_model.npz",
    "inversion_model_file": "init_model.npz",
    "f0": 5.0,
    "shots": 3,
    "nt_samples": 3000,
    "checkpoint_segments": 10,
    "iterations_short": 10,
    "iterations_long": 100,
    "optimizer": "adam",
    "lr": 10.0,
    "scheduler_step_size": 200,
    "scheduler_gamma": 0.75,
    "misfit": "legacy-l2",
    "waveform_normalize": True,
    "auto_update_rho": True,
    "gradient_processor": "legacy",
}
INVERSION_CONFIG

## 3. Model Definitions

`true_model.npz` is used only to synthesize observed data. `init_model.npz` is the starting model for inversion.

In [ ]:
def model_summary(model_file: str):
    model_npz = load_npz(CASE_DIR / "data" / "model" / model_file)
    vp = np.asarray(model_npz["vp"])
    rho = np.asarray(model_npz["rho"])
    return {
        "model_file": model_file,
        "nx": int(model_npz["nx"]),
        "nz": int(model_npz["nz"]),
        "dx": float(model_npz["dx"]),
        "dz": float(model_npz["dz"]),
        "vp_shape": list(vp.shape),
        "vp_min": float(vp.min()),
        "vp_max": float(vp.max()),
        "rho_min": float(rho.min()),
        "rho_max": float(rho.max()),
    }

true_model = model_summary(INVERSION_CONFIG["true_model_file"])
initial_model = model_summary(INVERSION_CONFIG["inversion_model_file"])
true_model, initial_model

## 4. Observation System And Wavelet Definition

The survey and source wavelet are rebuilt from the saved Marmousi2 observation metadata. The inversion stage uses the first `shots` sources and the first `nt_samples` time samples.

In [ ]:
obs_npz = load_npz(CASE_DIR / "data" / "waveform" / "obs_data.npz")
survey = build_survey(obs_npz, f0=INVERSION_CONFIG["f0"])
survey_summary = {
    "full_shots": survey.source.num,
    "used_shots": INVERSION_CONFIG["shots"],
    "receivers": survey.receiver.num,
    "full_nt": survey.source.nt,
    "used_nt_samples": INVERSION_CONFIG["nt_samples"],
    "dt": survey.source.dt,
    "f0": INVERSION_CONFIG["f0"],
    "src_x_range": [int(np.min(survey.source.get_loc()[:, 0])), int(np.max(survey.source.get_loc()[:, 0]))],
    "rcv_x_range": [int(np.min(survey.receiver.get_loc()[:, 0])), int(np.max(survey.receiver.get_loc()[:, 0]))],
}
survey_summary

## 5. Run Inversion Validation

The default cell is a dry run. Set `RUN_INVERSION_10 = True` to execute the 10-iteration validation.

In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False, iterations: int | None = None):
    command = [
        sys.executable,
        str(SCRIPT),
        stage,
        "--case-dir", str(CASE_DIR),
        "--device", INVERSION_CONFIG["device"],
        "--dtype", INVERSION_CONFIG["dtype"],
        "--f0", str(INVERSION_CONFIG["f0"]),
        "--inversion-model-file", INVERSION_CONFIG["inversion_model_file"],
        "--shots", str(INVERSION_CONFIG["shots"]),
        "--nt-samples", str(INVERSION_CONFIG["nt_samples"]),
        "--checkpoint-segments", str(INVERSION_CONFIG["checkpoint_segments"]),
        "--lr", str(INVERSION_CONFIG["lr"]),
        "--scheduler-step-size", str(INVERSION_CONFIG["scheduler_step_size"]),
        "--scheduler-gamma", str(INVERSION_CONFIG["scheduler_gamma"]),
        "--gradient-processor", INVERSION_CONFIG["gradient_processor"],
        "--output-root", str(OUTPUT_ROOT),
    ]
    if iterations is not None:
        command.extend(["--iterations", str(iterations)])
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])

run_stage("inversion10", dry_run=True, iterations=INVERSION_CONFIG["iterations_short"])

In [ ]:
RUN_INVERSION_10 = False

if RUN_INVERSION_10:
    inversion10_result = run_stage(
        "inversion10",
        dry_run=False,
        overwrite=True,
        iterations=INVERSION_CONFIG["iterations_short"],
    )
else:
    inversion10_result = {"status": "skipped", "reason": "set RUN_INVERSION_10=True"}

inversion10_result

## 6. Optional 100-Iteration Run

Only run this after the 10-iteration loss curve and model update are reasonable.

In [ ]:
RUN_INVERSION_100 = False

if RUN_INVERSION_100:
    inversion100_result = run_stage(
        "inversion100",
        dry_run=False,
        overwrite=True,
        iterations=INVERSION_CONFIG["iterations_long"],
    )
else:
    inversion100_result = {"status": "skipped", "reason": "set RUN_INVERSION_100=True"}

inversion100_result